In [5]:
import os
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix, 
                            roc_auc_score, roc_curve, accuracy_score,
                            precision_score, recall_score, f1_score)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

TRAIN_FEATURES_CSV = "galaxy_features_train.csv"  # UPDATE THIS PATH
TRAIN_LABELS_CSV = "trainlabel.csv"
TEST_FEATURES_CSV = "galaxy_features_test.csv"
TEST_LABELS_CSV = "testlabel.csv"

# SVM Configuration (based on research best practices)
SVM_CONFIG = {
    'kernel': 'rbf',  # Radial Basis Function kernel
    'C': 78,  # Regularization parameter (from reference)
    'gamma': 'scale',  # Kernel coefficient
    'class_weight': 'balanced',  # Handle class imbalance
    'max_iter': -1,  # No iteration limit
    'random_state': 42
}

# Cross-validation settings
CV_FOLDS = 5
RANDOM_STATE = 42

print("="*80)
print("SVM GALAXY MORPHOLOGY CLASSIFIER")
print("Binary Classification: Barred (1) vs Unbarred (0) Spiral Galaxies")
print("="*80)

# ============================================================================
# PART 1: DATA LOADING WITH STRICT SEPARATION
# ============================================================================

def load_data(features_csv, labels_csv, dataset_name="Dataset"):
    """
    Load features and labels with strict validation
    """
    print(f"\n{'='*80}")
    print(f"LOADING {dataset_name.upper()}")
    print("="*80)
    
    # Load features
    if not os.path.exists(features_csv):
        raise FileNotFoundError(f"Features file not found: {features_csv}")
    
    features_df = pd.read_csv(features_csv)
    print(f"✓ Loaded features from {features_csv}")
    print(f"  Shape: {features_df.shape}")
    
    # Load labels
    if not os.path.exists(labels_csv):
        raise FileNotFoundError(f"Labels file not found: {labels_csv}")
    
    labels_df = pd.read_csv(labels_csv)
    print(f"✓ Loaded labels from {labels_csv}")
    print(f"  Shape: {labels_df.shape}")
    
    # Check for and remove NaN labels
    nan_count = labels_df['Label'].isna().sum()
    if nan_count > 0:
        print(f"⚠ Warning: Found {nan_count} galaxies with missing labels - removing them")
        labels_df = labels_df.dropna(subset=['Label'])
        print(f"  After removing NaN: {len(labels_df)} galaxies remain")
    
    # Ensure labels are integers
    labels_df['Label'] = labels_df['Label'].astype(int)
    
    # Validate label values
    unique_labels = labels_df['Label'].unique()
    if not set(unique_labels).issubset({0, 1}):
        print(f"⚠ Warning: Found unexpected label values: {unique_labels}")
        print(f"  Expected only 0 (Unbarred) and 1 (Barred)")
        # Filter to only valid labels
        labels_df = labels_df[labels_df['Label'].isin([0, 1])]
        print(f"  After filtering: {len(labels_df)} galaxies with valid labels")
    
    # Extract galaxy names from filename column
    if 'filename' in features_df.columns:
        # Extract target name from filename
        # Example: norm_resized_NGC_0055.fits -> NGC 0055
        def extract_target_name(filename):
            # Remove prefix and suffix
            name = filename.replace('norm_resized_', '').replace('.fits', '')
            # Replace underscores with spaces
            name = name.replace('_', ' ')
            return name
        
        features_df['Target Name'] = features_df['filename'].apply(extract_target_name)
    else:
        raise ValueError("Features CSV must have 'filename' column")
    
    # Debug: Show first few target names
    print(f"\n  Sample target names from features:")
    print(f"    {features_df['Target Name'].head(3).tolist()}")
    print(f"  Sample target names from labels:")
    print(f"    {labels_df['Target Name'].head(3).tolist()}")
    
    # Merge features with labels
    merged_df = pd.merge(
        features_df, 
        labels_df, 
        on='Target Name', 
        how='inner'
    )
    
    if len(merged_df) == 0:
        print("\n❌ ERROR: No matching galaxies found!")
        print(f"\n  Features file has {len(features_df)} galaxies")
        print(f"  Labels file has {len(labels_df)} galaxies")
        print(f"\n  Sample names from features (first 5):")
        for name in features_df['Target Name'].head().tolist():
            print(f"    '{name}'")
        print(f"\n  Sample names from labels (first 5):")
        for name in labels_df['Target Name'].head().tolist():
            print(f"    '{name}'")
        
        # Check if any names match after cleaning
        features_set = set(features_df['Target Name'].str.strip().str.upper())
        labels_set = set(labels_df['Target Name'].str.strip().str.upper())
        common = features_set.intersection(labels_set)
        print(f"\n  Common names (case-insensitive): {len(common)}")
        if len(common) > 0:
            print(f"    Examples: {list(common)[:3]}")
        
        raise ValueError(f"No matching galaxies found between features and labels!")
    
    print(f"✓ Merged dataset: {len(merged_df)} galaxies")
    
    # Extract features (exclude filename and target name columns)
    feature_columns = [col for col in features_df.columns 
                      if col not in ['filename', 'Target Name']]
    
    X = merged_df[feature_columns].values
    y = merged_df['Label'].values
    galaxy_names = merged_df['Target Name'].values
    
    # Final validation: Check for NaN in features and labels
    if np.isnan(y).any():
        nan_indices = np.where(np.isnan(y))[0]
        print(f"⚠ Critical: Found {len(nan_indices)} NaN values in labels after merge!")
        print(f"  Affected galaxies: {galaxy_names[nan_indices][:5]}")
        # Remove samples with NaN labels
        valid_mask = ~np.isnan(y)
        X = X[valid_mask]
        y = y[valid_mask]
        galaxy_names = galaxy_names[valid_mask]
        print(f"  After removing NaN labels: {len(y)} samples remain")
    
    # Ensure y is integer type
    y = y.astype(int)
    
    # Check for missing values
    if np.isnan(X).any():
        print(f"⚠ Warning: Found {np.isnan(X).sum()} NaN values in features")
        print("  Replacing NaN with column mean...")
        col_mean = np.nanmean(X, axis=0)
        inds = np.where(np.isnan(X))
        X[inds] = np.take(col_mean, inds[1])
    
    # Class distribution
    unique, counts = np.unique(y, return_counts=True)
    print(f"\n  Class Distribution:")
    print(f"    Unbarred (0): {counts[0]} ({counts[0]/len(y)*100:.1f}%)")
    print(f"    Barred (1):   {counts[1]} ({counts[1]/len(y)*100:.1f}%)")
    
    print(f"\n  Feature Set: {len(feature_columns)} features")
    print(f"    {', '.join(feature_columns[:5])}, ...")
    
    return X, y, galaxy_names, feature_columns

# Load training data
X_train, y_train, train_names, feature_names = load_data(
    TRAIN_FEATURES_CSV, 
    TRAIN_LABELS_CSV, 
    "Training Set"
)

# Load test data (COMPLETELY SEPARATE - NO LEAKAGE)
X_test, y_test, test_names, _ = load_data(
    TEST_FEATURES_CSV, 
    TEST_LABELS_CSV, 
    "Test Set (Held-Out)"
)

# ============================================================================
# CRITICAL: DATA LEAKAGE CHECK
# ============================================================================

print(f"\n{'='*80}")
print("DATA LEAKAGE CHECK")
print("="*80)

train_set = set(train_names)
test_set = set(test_names)
overlap = train_set.intersection(test_set)

if len(overlap) > 0:
    print(f"❌ CRITICAL ERROR: {len(overlap)} galaxies appear in BOTH sets!")
    print(f"   Overlapping galaxies: {list(overlap)[:5]}")
    raise ValueError("DATA LEAKAGE DETECTED! Fix dataset split!")
else:
    print(f"✓ No data leakage detected")
    print(f"  Training set: {len(train_set)} unique galaxies")
    print(f"  Test set: {len(test_set)} unique galaxies")
    print(f"  Total unique: {len(train_set) + len(test_set)} galaxies")

# ============================================================================
# PART 2: FEATURE SCALING (FIT ON TRAINING ONLY)
# ============================================================================

print(f"\n{'='*80}")
print("FEATURE SCALING")
print("="*80)

# Initialize scaler
scaler = StandardScaler()

# Fit scaler ONLY on training data
X_train_scaled = scaler.fit_transform(X_train)
print(f"✓ Scaler fitted on training data")
print(f"  Mean: {scaler.mean_[:3]} ...")
print(f"  Std: {scaler.scale_[:3]} ...")

# Transform test data using training scaler (NO LEAKAGE)
X_test_scaled = scaler.transform(X_test)
print(f"✓ Test data scaled using training parameters (no leakage)")

# ============================================================================
# PART 3: SVM MODEL TRAINING
# ============================================================================

print(f"\n{'='*80}")
print("SVM MODEL TRAINING")
print("="*80)

# Create SVM classifier with optimal parameters
svm_model = SVC(**SVM_CONFIG, probability=True, verbose=False)

print(f"SVM Configuration:")
print(f"  Kernel: {SVM_CONFIG['kernel']}")
print(f"  C (Regularization): {SVM_CONFIG['C']}")
print(f"  Gamma: {SVM_CONFIG['gamma']}")
print(f"  Class Weight: {SVM_CONFIG['class_weight']}")
print(f"  Max Iterations: Unlimited")

# Train the model
print(f"\nTraining SVM on {len(X_train_scaled)} galaxies...")
svm_model.fit(X_train_scaled, y_train)
print(f"✓ SVM training completed")
print(f"  Support vectors: {svm_model.n_support_}")
print(f"    Class 0: {svm_model.n_support_[0]}")
print(f"    Class 1: {svm_model.n_support_[1]}")

# ============================================================================
# PART 4: K-FOLD CROSS-VALIDATION (ON TRAINING SET ONLY)
# ============================================================================

print(f"\n{'='*80}")
print("K-FOLD CROSS-VALIDATION (Training Set Only)")
print("="*80)

cv_scores = []
cv_precisions = []
cv_recalls = []
cv_f1s = []

skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

print(f"Performing {CV_FOLDS}-fold stratified cross-validation...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_scaled, y_train), 1):
    # Split data
    X_fold_train = X_train_scaled[train_idx]
    y_fold_train = y_train[train_idx]
    X_fold_val = X_train_scaled[val_idx]
    y_fold_val = y_train[val_idx]
    
    # Train fold model
    fold_model = SVC(**SVM_CONFIG, probability=True, verbose=False)
    fold_model.fit(X_fold_train, y_fold_train)
    
    # Predict
    y_fold_pred = fold_model.predict(X_fold_val)
    
    # Calculate metrics
    acc = accuracy_score(y_fold_val, y_fold_pred)
    prec = precision_score(y_fold_val, y_fold_pred, zero_division=0)
    rec = recall_score(y_fold_val, y_fold_pred, zero_division=0)
    f1 = f1_score(y_fold_val, y_fold_pred, zero_division=0)
    
    cv_scores.append(acc)
    cv_precisions.append(prec)
    cv_recalls.append(rec)
    cv_f1s.append(f1)
    
    print(f"  Fold {fold}: Acc={acc:.4f}, Prec={prec:.4f}, Rec={rec:.4f}, F1={f1:.4f}")

print(f"\nCross-Validation Results (Mean ± Std):")
print(f"  Accuracy:  {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")
print(f"  Precision: {np.mean(cv_precisions):.4f} ± {np.std(cv_precisions):.4f}")
print(f"  Recall:    {np.mean(cv_recalls):.4f} ± {np.std(cv_recalls):.4f}")
print(f"  F1-Score:  {np.mean(cv_f1s):.4f} ± {np.std(cv_f1s):.4f}")

# ============================================================================
# PART 5: HYPERPARAMETER TUNING (OPTIONAL)
# ============================================================================

print(f"\n{'='*80}")
print("HYPERPARAMETER TUNING (GridSearchCV)")
print("="*80)

# Define parameter grid
param_grid = {
    'C': [10, 50, 78, 100, 150],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1],
    'kernel': ['rbf', 'poly']
}

print(f"Testing {np.prod([len(v) for v in param_grid.values()])} parameter combinations...")

grid_search = GridSearchCV(
    SVC(class_weight='balanced', probability=True, random_state=RANDOM_STATE),
    param_grid,
    cv=StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE),
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)

grid_search.fit(X_train_scaled, y_train)

print(f"\n✓ Grid Search completed")
print(f"  Best Parameters: {grid_search.best_params_}")
print(f"  Best CV Accuracy: {grid_search.best_score_:.4f}")

# Use best model for final predictions
best_svm_model = grid_search.best_estimator_

# ============================================================================
# PART 6: TRAINING SET EVALUATION
# ============================================================================

def evaluate_model(model, X, y, dataset_name, galaxy_names=None):
    """
    Comprehensive model evaluation
    """
    print(f"\n{'='*80}")
    print(f"EVALUATION: {dataset_name.upper()}")
    print("="*80)
    
    # Predictions
    y_pred = model.predict(X)
    y_pred_proba = model.predict_proba(X)[:, 1]
    
    # Calculate metrics
    accuracy = accuracy_score(y, y_pred)
    precision = precision_score(y, y_pred, zero_division=0)
    recall = recall_score(y, y_pred, zero_division=0)
    f1 = f1_score(y, y_pred, zero_division=0)
    
    try:
        auc = roc_auc_score(y, y_pred_proba)
        print(f"\n✓ Overall Metrics:")
        print(f"  Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
        print(f"  Precision: {precision:.4f}")
        print(f"  Recall:    {recall:.4f}")
        print(f"  F1-Score:  {f1:.4f}")
        print(f"  AUC-ROC:   {auc:.4f}")
    except ValueError:
        print(f"\n✓ Overall Metrics:")
        print(f"  Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
        print(f"  Precision: {precision:.4f}")
        print(f"  Recall:    {recall:.4f}")
        print(f"  F1-Score:  {f1:.4f}")
        print(f"  AUC-ROC:   N/A (only one class present)")
        auc = None
    
    # Classification report
    print(f"\nClassification Report:")
    print(classification_report(y, y_pred, 
                                target_names=['Unbarred (0)', 'Barred (1)'],
                                digits=4,
                                zero_division=0))
    
    # Confusion matrix
    cm = confusion_matrix(y, y_pred)
    print(f"Confusion Matrix:")
    print(f"\n                Predicted")
    print(f"              Unbarred  Barred")
    print(f"  Actual")
    print(f"  Unbarred     {cm[0,0]:4d}     {cm[0,1]:4d}")
    print(f"  Barred       {cm[1,0]:4d}     {cm[1,1]:4d}")
    
    # Class-wise accuracy
    if cm[0,0] + cm[0,1] > 0:
        unbarred_acc = cm[0,0] / (cm[0,0] + cm[0,1])
        print(f"\n  Class-wise Accuracy:")
        print(f"    Unbarred: {unbarred_acc:.4f} ({unbarred_acc*100:.2f}%)")
    if cm[1,0] + cm[1,1] > 0:
        barred_acc = cm[1,1] / (cm[1,0] + cm[1,1])
        print(f"    Barred:   {barred_acc:.4f} ({barred_acc*100:.2f}%)")
    
    # Misclassified examples
    misclassified = np.where(y != y_pred)[0]
    if len(misclassified) > 0 and galaxy_names is not None:
        print(f"\n  Misclassified Galaxies: {len(misclassified)}")
        print(f"    Examples (first 5):")
        for idx in misclassified[:5]:
            print(f"      {galaxy_names[idx]}: True={y[idx]}, Pred={y_pred[idx]} (prob={y_pred_proba[idx]:.3f})")
    
    return {
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc': auc,
        'confusion_matrix': cm
    }

# Evaluate on training set (to check for overfitting)
train_results = evaluate_model(
    best_svm_model, 
    X_train_scaled, 
    y_train, 
    "Training Set",
    train_names
)

# ============================================================================
# PART 7: TEST SET EVALUATION (FINAL VALIDATION - NO LEAKAGE)
# ============================================================================

test_results = evaluate_model(
    best_svm_model, 
    X_test_scaled, 
    y_test, 
    "Test Set (Held-Out - Final Validation)",
    test_names
)

# ============================================================================
# PART 8: FEATURE IMPORTANCE ANALYSIS
# ============================================================================

print(f"\n{'='*80}")
print("FEATURE IMPORTANCE ANALYSIS")
print("="*80)

# For linear kernel, we can get feature weights directly
# For RBF kernel, we use permutation importance
from sklearn.inspection import permutation_importance

print("Computing permutation importance (this may take a moment)...")

perm_importance = permutation_importance(
    best_svm_model, 
    X_test_scaled, 
    y_test,
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# Sort features by importance
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': perm_importance.importances_mean,
    'Std': perm_importance.importances_std
}).sort_values('Importance', ascending=False)

print(f"\nTop 10 Most Important Features:")
print(importance_df.head(10).to_string(index=False))

# ============================================================================
# PART 9: VISUALIZATION
# ============================================================================

print(f"\n{'='*80}")
print("GENERATING VISUALIZATIONS")
print("="*80)

# Create figure with subplots
fig = plt.figure(figsize=(16, 12))

# 1. Confusion Matrix - Training
ax1 = plt.subplot(2, 3, 1)
sns.heatmap(train_results['confusion_matrix'], annot=True, fmt='d', cmap='Blues',
            xticklabels=['Unbarred', 'Barred'],
            yticklabels=['Unbarred', 'Barred'])
ax1.set_title('Confusion Matrix - Training Set', fontsize=14, fontweight='bold')
ax1.set_ylabel('True Label', fontsize=12)
ax1.set_xlabel('Predicted Label', fontsize=12)

# 2. Confusion Matrix - Test
ax2 = plt.subplot(2, 3, 2)
sns.heatmap(test_results['confusion_matrix'], annot=True, fmt='d', cmap='Greens',
            xticklabels=['Unbarred', 'Barred'],
            yticklabels=['Unbarred', 'Barred'])
ax2.set_title('Confusion Matrix - Test Set', fontsize=14, fontweight='bold')
ax2.set_ylabel('True Label', fontsize=12)
ax2.set_xlabel('Predicted Label', fontsize=12)

# 3. ROC Curve
ax3 = plt.subplot(2, 3, 3)
if test_results['auc'] is not None:
    fpr, tpr, _ = roc_curve(y_test, test_results['y_pred_proba'])
    ax3.plot(fpr, tpr, linewidth=2, label=f'SVM (AUC = {test_results["auc"]:.3f})')
    ax3.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
    ax3.set_xlabel('False Positive Rate', fontsize=12)
    ax3.set_ylabel('True Positive Rate', fontsize=12)
    ax3.set_title('ROC Curve - Test Set', fontsize=14, fontweight='bold')
    ax3.legend(fontsize=10)
    ax3.grid(True, alpha=0.3)

# 4. Feature Importance
ax4 = plt.subplot(2, 3, 4)
top_features = importance_df.head(10)
ax4.barh(range(len(top_features)), top_features['Importance'])
ax4.set_yticks(range(len(top_features)))
ax4.set_yticklabels(top_features['Feature'])
ax4.set_xlabel('Importance Score', fontsize=12)
ax4.set_title('Top 10 Feature Importance', fontsize=14, fontweight='bold')
ax4.invert_yaxis()
ax4.grid(True, alpha=0.3, axis='x')

# 5. Performance Comparison
ax5 = plt.subplot(2, 3, 5)
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
train_scores = [train_results['accuracy'], train_results['precision'], 
                train_results['recall'], train_results['f1']]
test_scores = [test_results['accuracy'], test_results['precision'], 
               test_results['recall'], test_results['f1']]

x = np.arange(len(metrics))
width = 0.35

ax5.bar(x - width/2, train_scores, width, label='Training', alpha=0.8)
ax5.bar(x + width/2, test_scores, width, label='Test', alpha=0.8)
ax5.set_ylabel('Score', fontsize=12)
ax5.set_title('Performance Metrics Comparison', fontsize=14, fontweight='bold')
ax5.set_xticks(x)
ax5.set_xticklabels(metrics, rotation=45)
ax5.legend(fontsize=10)
ax5.grid(True, alpha=0.3, axis='y')
ax5.set_ylim([0, 1.05])

# 6. Cross-Validation Scores
ax6 = plt.subplot(2, 3, 6)
cv_metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
cv_means = [np.mean(cv_scores), np.mean(cv_precisions), 
            np.mean(cv_recalls), np.mean(cv_f1s)]
cv_stds = [np.std(cv_scores), np.std(cv_precisions), 
           np.std(cv_recalls), np.std(cv_f1s)]

ax6.bar(cv_metrics, cv_means, alpha=0.7, yerr=cv_stds, capsize=5)
ax6.set_ylabel('Score', fontsize=12)
ax6.set_title(f'{CV_FOLDS}-Fold Cross-Validation Results', fontsize=14, fontweight='bold')
ax6.set_xticklabels(cv_metrics, rotation=45)
ax6.grid(True, alpha=0.3, axis='y')
ax6.set_ylim([0, 1.05])

plt.tight_layout()
plt.savefig('svm_galaxy_classification_results.png', dpi=300, bbox_inches='tight')
print("✓ Visualization saved as 'svm_galaxy_classification_results.png'")
plt.close()

# ============================================================================
# PART 10: SAVE RESULTS
# ============================================================================

print(f"\n{'='*80}")
print("SAVING RESULTS")
print("="*80)

# Save predictions
predictions_df = pd.DataFrame({
    'Target_Name': test_names,
    'True_Label': y_test,
    'Predicted_Label': test_results['y_pred'],
    'Prediction_Probability': test_results['y_pred_proba'],
    'Correct': y_test == test_results['y_pred']
})

predictions_df.to_csv('svm_predictions.csv', index=False)
print("✓ Predictions saved to 'svm_predictions.csv'")

# Save feature importance
importance_df.to_csv('feature_importance.csv', index=False)
print("✓ Feature importance saved to 'feature_importance.csv'")

# Save model summary
summary = {
    'Model': 'Support Vector Machine (SVM)',
    'Training_Samples': len(X_train),
    'Test_Samples': len(X_test),
    'Features': len(feature_names),
    'Best_Parameters': str(grid_search.best_params_),
    'Training_Accuracy': f"{train_results['accuracy']:.4f}",
    'Test_Accuracy': f"{test_results['accuracy']:.4f}",
    'Test_Precision': f"{test_results['precision']:.4f}",
    'Test_Recall': f"{test_results['recall']:.4f}",
    'Test_F1': f"{test_results['f1']:.4f}",
    'Test_AUC': f"{test_results['auc']:.4f}" if test_results['auc'] else 'N/A',
    'CV_Accuracy_Mean': f"{np.mean(cv_scores):.4f}",
    'CV_Accuracy_Std': f"{np.std(cv_scores):.4f}"
}

summary_df = pd.DataFrame([summary])
summary_df.to_csv('svm_model_summary.csv', index=False)
print("✓ Model summary saved to 'svm_model_summary.csv'")

# ============================================================================
# FINAL SUMMARY
# ============================================================================

print(f"\n{'='*80}")
print("FINAL SUMMARY")
print("="*80)

print(f"\n✓ SVM Galaxy Classification Complete!")
print(f"\n  Dataset:")
print(f"    Training: {len(X_train)} galaxies")
print(f"    Test: {len(X_test)} galaxies (held-out, no leakage)")
print(f"    Features: {len(feature_names)}")

print(f"\n  Best Model Configuration:")
for key, value in grid_search.best_params_.items():
    print(f"    {key}: {value}")

print(f"\n  Cross-Validation ({CV_FOLDS}-fold):")
print(f"    Accuracy: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")

print(f"\n  Final Test Performance:")
print(f"    Accuracy:  {test_results['accuracy']:.4f} ({test_results['accuracy']*100:.2f}%)")
print(f"    Precision: {test_results['precision']:.4f}")
print(f"    Recall:    {test_results['recall']:.4f}")
print(f"    F1-Score:  {test_results['f1']:.4f}")
if test_results['auc']:
    print(f"    AUC-ROC:   {test_results['auc']:.4f}")

print(f"\n  Files Generated:")
print(f"    ✓ svm_galaxy_classification_results.png")
print(f"    ✓ svm_predictions.csv")
print(f"    ✓ feature_importance.csv")
print(f"    ✓ svm_model_summary.csv")

print(f"\n{'='*80}")
print("Classification pipeline completed successfully!")
print("="*80)

SVM GALAXY MORPHOLOGY CLASSIFIER
Binary Classification: Barred (1) vs Unbarred (0) Spiral Galaxies

LOADING TRAINING SET
✓ Loaded features from galaxy_features_train.csv
  Shape: (467, 14)
✓ Loaded labels from trainlabel.csv
  Shape: (466, 2)
⚠ Warning: Found 2 galaxies with missing labels - removing them
  After removing NaN: 464 galaxies remain

  Sample target names from features:
    ['ARP 314 NED01', 'ARP 314 NED02', 'Centaurus A']
  Sample target names from labels:
    ['NGC 0024', 'NGC 0099', 'NGC 0115']
✓ Merged dataset: 464 galaxies

  Class Distribution:
    Unbarred (0): 213 (45.9%)
    Barred (1):   251 (54.1%)

  Feature Set: 13 features
    DA, DV, Asymmetry, Clumpiness, Concentration, ...

LOADING TEST SET (HELD-OUT)
✓ Loaded features from galaxy_features_test.csv
  Shape: (117, 14)
✓ Loaded labels from testlabel.csv
  Shape: (117, 2)

  Sample target names from features:
    ['CGCG 023-019', 'ESO 034-G013', 'ESO 291-G009']
  Sample target names from labels:
    ['NGC 00